In [ ]:
import anthropic
import pandas as pd
import re
import time
from pathlib import Path
import os

In [ ]:
MODEL = "claude-opus-4-5-20251101"
API_KEY = ""
DELAY_BETWEEN_CALLS = 1.0
DIMINISHMENT_THRESHOLD = 0.05

In [ ]:
def load_speech(filepath):
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f"Speech file not found: {filepath}")
    if path.suffix.lower() != ".txt":
        raise ValueError(f"Expected a .txt file, got: {path.suffix}")

    raw = path.read_text(encoding="utf-8", errors="replace")
    text = raw.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

In [ ]:
def build_prompt(text: str) -> str:
    return f"""You are a political language analyst. Score the following political speech on four dimensions.

---

AGENCY SCORE (0.0 to 1.0)
How much does the speaker present themselves or their party as the active, capable driver of change?
0.0 = entirely passive, reactive, or victim-framed language
1.0 = strong first-person ownership, decisive verbs, clear subject-verb-object construction
Consider: use of "I will", "we did", active vs passive voice, first-person subjects.

DIMINISHMENT SCORE (0.0 to 1.0)
How much does the speaker frame the opponent as weak, incompetent, irrelevant, ridiculous, or non-threatening?
0.0 = opponent treated with full seriousness or not mentioned
1.0 = opponent systematically belittled, mocked, dismissed, or trivialized
Consider: minimizing language, sarcasm, nicknames, dismissive tone, rhetorical deflation.

DOMINANCE SCORE (0.0 to 1.0)
How much does the speaker project strength, superiority, and control over the political landscape?
0.0 = humble, collaborative, or deferential tone throughout
1.0 = strong assertion of superiority, contempt for opponents, hierarchical framing
Consider: disgust language, alpha-framing, assertions of inevitable victory, contempt.

MORAL FRAMING
Which moral frame dominates the speech?
- Authority/Dominance: order, strength, hierarchy, loyalty, control, decisive leadership
- Harm/Victim: protection, care, suffering, injustice, need for rescue or relief
Choose exactly one.

---

Do not explain your reasoning. Do not list examples. Do not use headers or bullet points.
Your entire response must be exactly one line in this format:
Agency: X.XX, Diminishment: X.XX, Dominance: X.XX, Moral Framing: <Authority/Dominance or Harm/Victim>

Text to score:
\"\"\"{text}\"\"\"
"""


def call_api(text: str) -> str:
    client = anthropic.Anthropic(api_key=API_KEY)
    prompt = build_prompt(text)

    message = client.messages.create(
        model=MODEL,
        max_tokens=128,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text

In [ ]:
def parse_response(raw: str, threshold: float = DIMINISHMENT_THRESHOLD) -> dict:
    agency    = re.search(r"Agency:\s*([0-9.]+)",       raw, re.IGNORECASE)
    diminish  = re.search(r"Diminishment:\s*([0-9.]+)", raw, re.IGNORECASE)
    dominance = re.search(r"Dominance:\s*([0-9.]+)",    raw, re.IGNORECASE)
    moral     = re.search(r"Moral Framing:\s*(.+)",     raw, re.IGNORECASE)

    parse_error = None
    missing = [name for name, m in [
        ("Agency", agency), ("Diminishment", diminish),
        ("Dominance", dominance), ("Moral Framing", moral)
    ] if not m]
    if missing:
        parse_error = f"Could not parse: {', '.join(missing)}"

    def to_float(m):
        return round(float(m.group(1)), 4) if m else None

    diminishment_score = to_float(diminish)

    return {
        "agency_score":             to_float(agency),
        "diminishment_score":       diminishment_score,
        "dominance_score":          to_float(dominance),
        "moral_framing":            moral.group(1).strip() if moral else None,
        "making_opponent_small":    (diminishment_score > threshold)
                                    if diminishment_score is not None else None,
        "raw_response":             raw,
        "parse_error":              parse_error,
    }


def score_speech(speech_path: str) -> dict:
    speech = load_speech(speech_path)
    raw = call_api(speech)
    result = parse_response(raw)
    result["filename"] = Path(speech_path).name
    return result

In [ ]:
def score_all_speeches(speech_folders, output_path):
    results = []

    for folder in speech_folders:
        speaker = Path(folder).name
        for filename in os.listdir(folder):
            if filename.endswith(".txt"):
                speech_path = os.path.join(folder, filename)
                print(f"Scoring: {speech_path}")
                result = score_speech(speech_path)
                result["speaker"] = speaker
                results.append(result)
                time.sleep(DELAY_BETWEEN_CALLS)

    df = pd.DataFrame(results)
    df = df[["speaker", "filename", "agency_score", "diminishment_score",
             "dominance_score", "moral_framing", "making_opponent_small",
             "raw_response", "parse_error"]]
    df.to_excel(output_path, index=False)
    print(f"\nDone! Results saved to {output_path}")
    return df


folders = [
    "Zara-Ahsan-Thesis-Repository/speeches/andy beshear",
    "Zara-Ahsan-Thesis-Repository/speeches/Anthony Brown",
    "Zara-Ahsan-Thesis-Repository/speeches/Charlie Baker",
    "Zara-Ahsan-Thesis-Repository/speeches/Kris Kobach",
    "Zara-Ahsan-Thesis-Repository/speeches/Larry Hogan",
    "Zara-Ahsan-Thesis-Repository/speeches/Laura Kelly",
    "Zara-Ahsan-Thesis-Repository/speeches/Martha Coakley",
    "Zara-Ahsan-Thesis-Repository/speeches/matt bevin",
    "Zara-Ahsan-Thesis-Repository/speeches/Phil Scott",
    "Zara-Ahsan-Thesis-Repository/speeches/Sue Minter"
]

df_results = score_all_speeches(
    speech_folders=folders,
    output_path="dominance_frame_scores.xlsx"
)
df_results